In [1]:
import jax
import jax.numpy as jnp
from jax import lax, jit, vmap
from jax.experimental.pjit import pjit
from jax.experimental import mesh_utils
from jax.sharding import Mesh, PartitionSpec, PositionalSharding
from functools import partial
import time

# --- Configuration (Optimized for Stability & Speed)
MAX_RECURSION_DEPTH    = 1_000_000   # Testing highest recursion depth
OPTIMAL_DEPTH_STEP     = 250_000     # Breaking into manageable steps
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE             = 50_000_000  # Extreme scaling: 50M samples per batch

# Clamp bounds to avoid NaNs from overflow
VAL_CLAMP_LOW  = -100.0
VAL_CLAMP_HIGH =  100.0

# -------------------------------------------------------------------------
# 1) Dynamic pi & phi functions with scaling and stabilization.
# -------------------------------------------------------------------------
@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    return depth / (1 + jnp.log1p(depth + 1))

# -------------------------------------------------------------------------
# 2) Single 250K-step chunk (recursion) with clamping.
# -------------------------------------------------------------------------
@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    # Apply stabilization to the requested depth.
    depth = stabilize_depth(jnp.minimum(depth, MAX_RECURSION_DEPTH))

    def body_fn(i, val):
        pi_dyn  = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        # Clamp to avoid overflow/NaNs.
        safe_val = jnp.clip(val, VAL_CLAMP_LOW, VAL_CLAMP_HIGH)
        new_val = jnp.sin(safe_val * scale * pi_dyn) * jnp.exp(-safe_val / (phi_dyn + 1))
        return new_val

    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

# -------------------------------------------------------------------------
# 3) 8-Core CPU Sharding Setup using a device mesh.
# -------------------------------------------------------------------------
# Request CPU devices. If fewer than 8 are exposed, create a virtual mesh.
devices = jax.devices("cpu")
if len(devices) < 8:
    devices = mesh_utils.create_device_mesh((8,))  # Virtual 8-core mesh.
else:
    devices = devices[:8]

mesh = Mesh(devices, ("data",))
sharding = PositionalSharding(mesh.devices.flat)

# -------------------------------------------------------------------------
# 4) Batched Processing via pjit + vmap.
# -------------------------------------------------------------------------
# Here, we map our function across the batch dimension.
# Note: Our function dppu_with_dynamic_pi_phi is called inside vmap.
batched_dppu_processing = pjit(
    lambda arr: vmap(lambda xi: dppu_with_dynamic_pi_phi(xi, depth=OPTIMAL_DEPTH_STEP, scale_factor=0.5), in_axes=0)(arr),
    in_shardings=(sharding,),
    out_shardings=sharding,
)

# -------------------------------------------------------------------------
# 5) Adaptive Execution: Process in chunks.
# -------------------------------------------------------------------------
def process_with_larger_depths(x, total_depth):
    """Instead of running all at once, we execute in 250K-depth chunks."""
    iterations = total_depth // OPTIMAL_DEPTH_STEP
    for _ in range(iterations):
        x = dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP)
    return x

# -------------------------------------------------------------------------
# 6) Warm-up and Benchmarking.
# -------------------------------------------------------------------------
print("Allocating batch_input...")
batch_input = jnp.linspace(0, 10, BATCH_SIZE)
# Place batch_input on the TPU mesh (here, our CPU mesh) using our sharding.
batch_input = jax.device_put(batch_input, sharding)

# Warm-up compile: compile for 250K steps.
_ = dppu_with_dynamic_pi_phi(jnp.ones((BATCH_SIZE,)), depth=OPTIMAL_DEPTH_STEP)

# Run execution for each total_depth and time it.
for depth in [250_000, 500_000, 1_000_000]:
    start_time = time.time()
    output_batch = process_with_larger_depths(batch_input, depth)
    end_time = time.time()
    print(f"✅ Batch Output Shape (Depth={depth}):", output_batch.shape)
    print(f"🔥 Execution Time: {end_time - start_time:.6f} sec")

NUM_TRIALS = 2  # Reduce trials to avoid overload

for depth in [250_000, 500_000, 1_000_000]:
    times = []
    for _ in range(NUM_TRIALS):
        start = time.time()
        result = process_with_larger_depths(jnp.ones((BATCH_SIZE,)), depth)
        _ = jax.device_get(result)
        end = time.time()
        times.append(end - start)
    avg_time = sum(times) / len(times)
    print(f"\n🔥 CPU Benchmark (Depth={depth}, Batch={BATCH_SIZE})")
    print(f"Avg: {avg_time:.6f} sec | Min: {min(times):.6f} sec | Max: {max(times):.6f} sec")

# -------------------------------------------------------------------------
# 7) Investigate XLA Compilation Stability.
# -------------------------------------------------------------------------
compiled_fn_250k = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((BATCH_SIZE,)), depth=250_000)
compiled_fn_1M = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((BATCH_SIZE,)), depth=1_000_000)

print("\n🚀 XLA Compilation for Depth=250,000:")
print(compiled_fn_250k.as_text())

print("\n🚀 XLA Compilation for Depth=1,000,000:")
print(compiled_fn_1M.as_text())


Allocating batch_input...
✅ Batch Output Shape (Depth=250000): (50000000,)
🔥 Execution Time: 0.263318 sec
✅ Batch Output Shape (Depth=500000): (50000000,)
🔥 Execution Time: 0.001009 sec
✅ Batch Output Shape (Depth=1000000): (50000000,)
🔥 Execution Time: 0.001405 sec

🔥 CPU Benchmark (Depth=250000, Batch=50000000)
Avg: 254.083240 sec | Min: 131.064185 sec | Max: 377.102295 sec

🔥 CPU Benchmark (Depth=500000, Batch=50000000)
Avg: 262.005939 sec | Min: 262.004389 sec | Max: 262.007489 sec

🔥 CPU Benchmark (Depth=1000000, Batch=50000000)
Avg: 523.890676 sec | Min: 523.889131 sec | Max: 523.892221 sec

🚀 XLA Compilation for Depth=250,000:
module @jit_dppu_with_dynamic_pi_phi attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<50000000xf32> {mhlo.layout_mode = "default"}, %arg1: tensor<i32> {mhlo.layout_mode = "default"}) -> (tensor<50000000xf32> {jax.result_info = "", mhlo.layout_mode = "default"}) {
    %0 = call @dppu_with_dynam

/usr/local/lib/python3.11/dist-packages/jax/_src/core.py:701: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/jax/_src/core.py:701: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
